# Phase 10: Fine-tune workbench

The permanent sandbox for whatever model is currently being benchmarked.
Today that is **v4**: uniform mean of 5-seed ensembles over 6 stable PF configs
(beam, hold, NNLS, routing, grboth all retired by the lab). Structure:

1. **Control panel** — every lever in one place: harness, engine (including
   previously hardcoded facets), ensemble, post-processing.
2. **Engine** — superset-parameterized PF (defaults = v1-faithful) + beam kept
   runnable for future designs.
3. **Run** — hash-keyed resumable cache per config x mask mode: changing one
   lever never invalidates unrelated predictions.
4. **Evaluate** — pooled + per-well vs aligned references.
5. **A/B** — paired per-well comparison of two designs (deltas, win rate,
   sign test). Combiner-only variants are free from cache.
6. **Sensitivity scan** — one-lever-at-a-time quick look (1 seed, small sample).
7. **Export** — `model_card.json` + standalone `pf_engine.py` with an
   integrity test against a simulated test-style well. `submission.ipynb`
   vendors these.

## 1. Control panel

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import hashlib
import inspect
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from rogii_wellbore import clean  # noqa: E402

cfg = clean.load_config("../data/interim/clean_config.json")
CLEAN_DIR = Path("../data/interim/clean")
LAB_DIR = Path("../data/interim/lab_preds")
LAB_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("../data/interim/export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- HARNESS LEVERS ----------------
WELL_SAMPLE = None  # None = all wells; 250 = the nb9-comparable seeded sample
SAMPLE_SEED = 123
MASK_MODE = "flat"  # "flat" (0.73) | "realistic" (per-well U(0.67,0.80)) | float
N_FOLDS = 5  # used only when a stage fits something

# ---------------- ENGINE LEVERS (defaults = v1-faithful) ----------------
ENGINE_DEFAULTS = dict(
    N=500,
    spread=4.5,
    MOM=0.998,
    VN=0.002,
    PN=0.005,
    rate_win=30,
    gs_min=10.0,
    gs_max=60.0,
    RESAMP=0.5,
    RP=0.1,
    RR=0.001,
    init_rate_noise=0.01,
    lik_cap=600.0,
    min_dm=1.0,
    bound_margin=100.0,
    estimator="mean",  # "mean" | "trim" | "map"
    trim_q=0.1,  # used when estimator="trim"
    rate_est="median",  # "median" | "mean"
)

# ---------------- CURRENT DESIGN: v4 ----------------
V4_CONFIGS = {
    "sp2": dict(spread=2.0),
    "base": dict(),
    "sp2_vn004": dict(spread=2.0, VN=0.004),
    "resamp07": dict(RESAMP=0.7),
    "rp02": dict(RP=0.2),
    "sp2_mom999": dict(spread=2.0, MOM=0.999),
}
DESIGN = dict(
    name="v4",
    configs=V4_CONFIGS,
    seeds=(42, 7, 2024, 99, 1234),
    combiner="uniform",  # "uniform" | "median" | "trimmed"
    trimmed_q=0.2,  # used when combiner="trimmed"
    include_hold=False,  # add hold as a component
    postproc=dict(med_win=0, anchor_tau=0, slew_mult=0, hold_w=0.0),  # 0 = off
)
print(
    f"design: {DESIGN['name']} | {len(DESIGN['configs'])} configs x "
    f"{len(DESIGN['seeds'])} seeds | combiner={DESIGN['combiner']} | mask={MASK_MODE}"
)

## 2. Engine (superset-parameterized; beam kept for future designs)

In [ ]:
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


def tail_mask(n, frac):
    k = int(round(n * frac))
    m = np.zeros(n, bool)
    if k:
        m[n - k :] = True
    return m


def well_frac(wid, lo=0.67, hi=0.80):
    h = int(hashlib.md5(wid.encode()).hexdigest()[:8], 16)
    return lo + (hi - lo) * (h % 10_000) / 10_000.0


def mask_frac_for(wid):
    if MASK_MODE == "flat":
        return 0.73
    if MASK_MODE == "realistic":
        return well_frac(wid)
    return float(MASK_MODE)


def load_pair(wid):
    hz = (
        pd.read_csv(CLEAN_DIR / "train" / f"{wid}__horizontal_well.csv", dtype={"well_id": str})
        .sort_values("MD")
        .reset_index(drop=True)
    )
    tw = pd.read_csv(
        CLEAN_DIR / "train" / f"{wid}__typewell.csv", dtype={"well_id": str}
    ).reset_index(drop=True)
    return hz, tw


def prep_arrays(hz, tw):
    tw_s = tw.sort_values("TVT")
    return dict(
        twt=tw_s["TVT"].values.astype(float),
        twg=tw_s["GR"].ffill().bfill().values.astype(float),
        tvt=hz["TVT"].values.astype(float),
        Z=hz["Z"].values.astype(float),
        MD=hz["MD"].values.astype(float),
        gr=pd.Series(hz["GR"].values).interpolate(limit_direction="both").fillna(90.0).values,
    )


def run_pf(
    twt,
    twg,
    tvt,
    Z,
    MD,
    gr,
    kn,
    ev,
    N=500,
    spread=4.5,
    MOM=0.998,
    VN=0.002,
    PN=0.005,
    rate_win=30,
    gs_min=10.0,
    gs_max=60.0,
    seed=42,
    RESAMP=0.5,
    RP=0.1,
    RR=0.001,
    init_rate_noise=0.01,
    lik_cap=600.0,
    min_dm=1.0,
    bound_margin=100.0,
    estimator="mean",
    trim_q=0.1,
    rate_est="median",
):
    last = kn[-1]
    gs = float(np.clip(np.nanstd(gr[kn] - np.interp(tvt[kn], twt, twg)), gs_min, gs_max))
    tl = kn[-rate_win:]
    dt = np.diff(tvt[tl])
    dz = np.diff(Z[tl])
    dm = np.diff(MD[tl])
    ok = dm > 0
    if ok.sum() >= 3:
        r = (dt + dz)[ok] / dm[ok]
        ir = float(np.median(r)) if rate_est == "median" else float(np.mean(r))
    else:
        ir = 0.0
    rng = np.random.default_rng(seed)
    pos = (tvt[last] + Z[last]) + spread * rng.standard_normal(N)
    rate = ir + init_rate_noise * rng.standard_normal(N)
    w = np.ones(N) / N
    out = np.empty(len(ev))
    prev = MD[last]
    lo, hi = twt[0] - bound_margin, twt[-1] + bound_margin
    for i, idx in enumerate(ev):
        dmS = max(MD[idx] - prev, min_dm)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dmS + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - Z[idx], lo, hi)
        pos = tvt_p + Z[idx]
        g = gr[idx]
        if np.isfinite(g):
            d2 = ((g - np.interp(tvt_p, twt, twg)) / gs) ** 2
            w = w * np.maximum(np.exp(-0.5 * np.minimum(d2, lik_cap)), 1e-300)
            s = w.sum()
            w = w / s if s > 0 else np.ones(N) / N
        if 1.0 / np.sum(w * w) < RESAMP * N:
            ci = np.clip(
                np.searchsorted(np.cumsum(w), (np.arange(N) + rng.uniform(0, 1)) / N), 0, N - 1
            )
            pos = pos[ci] + RP * rng.standard_normal(N)
            rate = rate[ci] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
        if estimator == "mean":
            est = np.sum(w * (pos - Z[idx]))
        elif estimator == "map":
            est = (pos - Z[idx])[int(np.argmax(w))]
        else:
            tp = pos - Z[idx]
            order = np.argsort(tp)
            cw = np.cumsum(w[order])
            sel = (cw >= trim_q) & (cw <= 1 - trim_q)
            if not sel.any():
                sel = slice(None)
            ww = w[order][sel]
            est = float(np.sum(ww * tp[order][sel]) / max(ww.sum(), 1e-300))
        out[i] = est
        prev = MD[idx]
    return out


def run_beam(twt, twg, tvt, MD, gr, kn, ev, BS=10, mc=20.0, es=144.0, smooth=2):
    g = (
        pd.Series(gr).rolling(smooth, min_periods=1, center=True).mean().values
        if smooth > 1
        else gr
    )
    si = int(np.argmin(np.abs(twt - tvt[kn[-1]])))
    beams = {si: 0.0}
    hist = []
    for i in ev:
        gv = g[i]
        cand = {}
        for idx, cost in beams.items():
            for d in (-2, -1, 0, 1, 2):
                ni = idx + d
                if ni < 0 or ni >= len(twt):
                    continue
                tot = cost + (gv - twg[ni]) ** 2 / es + mc * abs(d)
                if ni not in cand or tot < cand[ni][0]:
                    cand[ni] = (tot, idx)
        top = sorted(cand.items(), key=lambda kv: kv[1][0])[:BS]
        hist.append({ni: p for ni, (c, p) in top})
        beams = {ni: c for ni, (c, p) in top}
    best = min(beams, key=beams.get)
    path = [best]
    for hm in reversed(hist[1:]):
        best = hm.get(best, best)
        path.append(best)
    return twt[np.array(path[::-1])]

## 3. Run (hash-keyed resumable cache)

Each (config x mask) gets its own cache folder keyed by a hash of the resolved
engine params; seeds live inside the file. Changing one lever recomputes only
what that lever touches.

In [ ]:
ALL_WELLS = sorted(
    p.name.split("__")[0] for p in (CLEAN_DIR / "train").glob("*__horizontal_well.csv")
)
if WELL_SAMPLE and len(ALL_WELLS) > WELL_SAMPLE:
    WELLS = sorted(np.random.default_rng(SAMPLE_SEED).choice(ALL_WELLS, WELL_SAMPLE, replace=False))
else:
    WELLS = ALL_WELLS
print(f"{len(WELLS)} wells | mask mode: {MASK_MODE}")


def resolved(cfg_overrides):
    p = dict(ENGINE_DEFAULTS)
    p.update(cfg_overrides)
    return p


def cfg_hash(cfg_overrides, seeds):
    payload = json.dumps(
        {"p": resolved(cfg_overrides), "seeds": list(seeds), "mask": str(MASK_MODE)}, sort_keys=True
    )
    return hashlib.md5(payload.encode()).hexdigest()[:10]


def compute_components(design, wells):
    comp = {}  # comp[name][wid] = prediction
    truth, hold = {}, {}
    for name, overrides in design["configs"].items():
        h = cfg_hash(overrides, design["seeds"])
        cdir = LAB_DIR / f"{name}_{h}"
        cdir.mkdir(exist_ok=True)
        comp[name] = {}
        t0 = time.time()
        done = 0
        for wid in wells:
            f = cdir / f"{wid}.npz"
            if f.exists():
                z = np.load(f)
                comp[name][wid] = z["pred"].astype(float)
                if wid not in truth:
                    truth[wid] = z["true"].astype(float)
                    hold[wid] = z["hold"].astype(float)
                continue
            try:
                hz, tw = load_pair(wid)
            except Exception:
                continue
            if "TVT" not in hz or hz["TVT"].isna().all():
                continue
            fr = mask_frac_for(wid)
            nrow = len(hz)
            if int(round(nrow * fr)) < 20 or nrow - int(round(nrow * fr)) < 20:
                continue
            A = prep_arrays(hz, tw)
            m = tail_mask(nrow, fr)
            kn = np.where(~m)[0]
            ev = np.where(m)[0]
            params = resolved(overrides)
            runs = [
                run_pf(
                    A["twt"], A["twg"], A["tvt"], A["Z"], A["MD"], A["gr"], kn, ev, seed=s, **params
                )
                for s in design["seeds"]
            ]
            pred = np.mean(runs, axis=0)
            tr = A["tvt"][ev]
            hd = np.full(len(ev), A["tvt"][kn[-1]])
            np.savez_compressed(
                f,
                pred=pred.astype(np.float32),
                true=tr.astype(np.float32),
                hold=hd.astype(np.float32),
            )
            comp[name][wid] = pred
            if wid not in truth:
                truth[wid] = tr
                hold[wid] = hd
            done += 1
            if done % 25 == 0:
                print(f"  [{name}] {done} computed [{(time.time()-t0)/60:.1f} min]")
        print(f"[{name}] ready ({len(comp[name])} wells)")
    return comp, truth, hold


COMP, TRUTH, HOLD = compute_components(DESIGN, WELLS)
COMMON = sorted(set.intersection(*[set(COMP[n]) for n in COMP]))
print(f"common wells across components: {len(COMMON)}")

## 4. Evaluate

In [ ]:
def combine(design, comp, wells):
    names = list(design["configs"])
    out = {}
    for w in wells:
        X = np.column_stack([comp[n][w] for n in names])
        if design["include_hold"]:
            X = np.column_stack([X, HOLD[w]])
        if design["combiner"] == "uniform":
            p = X.mean(1)
        elif design["combiner"] == "median":
            p = np.median(X, 1)
        else:
            q = design["trimmed_q"]
            lo, hi = np.quantile(X, [q, 1 - q], axis=1)
            Xc = np.clip(X, lo[:, None], hi[:, None])
            p = Xc.mean(1)
        pp = design["postproc"]
        if pp.get("hold_w", 0) > 0:
            p = (1 - pp["hold_w"]) * p + pp["hold_w"] * HOLD[w]
        if pp.get("anchor_tau", 0) > 0:
            t = np.arange(len(p), dtype=float)
            p = p + (HOLD[w][0] - p[0]) * np.exp(-t / pp["anchor_tau"])
        if pp.get("med_win", 0) > 1:
            p = pd.Series(p).rolling(int(pp["med_win"]), center=True, min_periods=1).median().values
        out[w] = p
    return out


def pooled(pv):
    e = np.concatenate([pv[w] - TRUTH[w] for w in pv])
    return float(np.sqrt(np.mean(e**2)))


def perwell(pv):
    return float(np.mean([rmse(pv[w], TRUTH[w]) for w in pv]))


PRED = combine(DESIGN, COMP, COMMON)
floor_pv = {w: HOLD[w] for w in COMMON}
print(f"{'stage':16s} {'pooled':>8s} {'per-well':>9s}")
print(f"{'floor':16s} {pooled(floor_pv):8.3f} {perwell(floor_pv):9.3f}")
print(f"{DESIGN['name']:16s} {pooled(PRED):8.3f} {perwell(PRED):9.3f}")
for n in DESIGN["configs"]:
    pv = {w: COMP[n][w] for w in COMMON}
    print(f"  comp {n:12s} {pooled(pv):8.3f} {perwell(pv):9.3f}")

V1_DIR = Path("../data/interim/pf_preds")
if V1_DIR.exists():
    V1K = ["pf_spread2", "pf_seedens", "pf_base", "pf_N300", "pf_spread8", "beam_cons"]
    v1 = {}
    for w in COMMON:
        f1 = V1_DIR / f"{w}.npz"
        if f1.exists():
            z1 = np.load(f1)
            if all(k in z1.files for k in V1K):
                v1[w] = np.mean([z1[k].astype(float) for k in V1K], axis=0)
    if v1:
        e = np.concatenate([v1[w] - TRUTH[w] for w in v1])
        print(
            f"\naligned v1 reference ({len(v1)} wells): pooled "
            f"{float(np.sqrt(np.mean(e**2))):.3f}"
        )
        if MASK_MODE != "flat":
            print("  (caution: v1 cache is flat-0.73; aligned only when MASK_MODE='flat')")

## 5. A/B — paired comparison of two designs

Combiner/post-proc variants reuse the cache (free); engine-param variants
compute their own cached components. Paired per-well deltas + sign test.

In [ ]:
import copy

from scipy.stats import binomtest

VARIANT = copy.deepcopy(DESIGN)
VARIANT["name"] = "v4_median"
VARIANT["combiner"] = "median"  # example free variant; edit at will

if VARIANT["configs"] == DESIGN["configs"] and VARIANT["seeds"] == DESIGN["seeds"]:
    COMP_V, wells_v = COMP, COMMON
else:
    COMP_V, TR_V, HD_V = compute_components(VARIANT, WELLS)
    wells_v = sorted(set(COMMON) & set.intersection(*[set(COMP_V[n]) for n in COMP_V]))

PRED_V = combine(VARIANT, COMP_V, wells_v)
da = np.array([rmse(PRED[w], TRUTH[w]) for w in wells_v])
db = np.array([rmse(PRED_V[w], TRUTH[w]) for w in wells_v])
delta = db - da
wins = int((delta < 0).sum())
losses = int((delta > 0).sum())
print(f"A = {DESIGN['name']}  pooled {pooled({w: PRED[w] for w in wells_v}):.3f}")
print(f"B = {VARIANT['name']}  pooled {pooled({w: PRED_V[w] for w in wells_v}):.3f}")
print(f"per-well delta (B-A): mean {delta.mean():+.3f} | median {np.median(delta):+.3f}")
print(
    f"B wins {wins}/{len(delta)} wells ({wins/len(delta):.0%}); "
    f"sign-test p={binomtest(wins, wins+losses).pvalue:.3f}"
    if wins + losses
    else ""
)

## 6. Sensitivity scan (quick look: 1 seed, small sample)

One global lever at a time, applied to every config. Deltas are directional
hints only — promote interesting levers to a proper A/B (5 seeds, full sample)
before believing them.

In [ ]:
SENS_WELLS = COMMON[: min(60, len(COMMON))]
SCAN = {
    "MOM": [0.996, 0.999],
    "VN": [0.001, 0.004],
    "PN": [0.002, 0.01],
    "gs_max": [40.0, 90.0],
    "rate_win": [15, 60],
    "estimator": ["trim", "map"],
    "lik_cap": [50.0],
}


def quick_score(overrides_global):
    errs = []
    for wid in SENS_WELLS:
        hz, tw = load_pair(wid)
        fr = mask_frac_for(wid)
        nrow = len(hz)
        A = prep_arrays(hz, tw)
        m = tail_mask(nrow, fr)
        kn = np.where(~m)[0]
        ev = np.where(m)[0]
        preds = []
        for name, ov in DESIGN["configs"].items():
            p = resolved(ov)
            p.update(overrides_global)
            preds.append(
                run_pf(A["twt"], A["twg"], A["tvt"], A["Z"], A["MD"], A["gr"], kn, ev, seed=42, **p)
            )
        errs.append(np.mean(preds, axis=0) - A["tvt"][ev])
    e = np.concatenate(errs)
    return float(np.sqrt(np.mean(e**2)))


t0 = time.time()
base_q = quick_score({})
print(f"quick base ({len(SENS_WELLS)} wells, 1 seed): {base_q:.3f}\n")
rows = []
for lever, vals in SCAN.items():
    for v in vals:
        s = quick_score({lever: v})
        rows.append((lever, v, s, s - base_q))
        print(f"  {lever:11s}={v!s:8s} {s:8.3f}  ({s-base_q:+.3f})")
print(f"\n[{(time.time()-t0)/60:.1f} min] negative delta = promising; verify via A/B")

## 7. Export — model card + standalone engine + integrity test

Writes `export/model_card.json` and `export/pf_engine.py` (numpy/pandas only,
no repo imports). The integrity test simulates a **test-style** well
(`TVT_input` with the NaN tail) and asserts the module's `predict_well`
reproduces the in-notebook prediction exactly — the contract
`submission.ipynb` relies on.

In [ ]:
MODEL_CARD = {
    "name": DESIGN["name"],
    "engine": "pf_v1_superset",
    "configs": {n: resolved(ov) for n, ov in DESIGN["configs"].items()},
    "seeds": list(DESIGN["seeds"]),
    "combiner": DESIGN["combiner"],
    "trimmed_q": DESIGN["trimmed_q"],
    "include_hold": DESIGN["include_hold"],
    "postproc": DESIGN["postproc"],
    "mask_mode_cv": str(MASK_MODE),
    "cv": {
        "wells": len(COMMON),
        "pooled": pooled(PRED),
        "per_well": perwell(PRED),
        "floor_pooled": pooled(floor_pv),
    },
    "inference_contract": "known zone = TVT_input.notna(); predictions for NaN rows",
}
with open(EXPORT_DIR / "model_card.json", "w") as f:
    json.dump(MODEL_CARD, f, indent=2, default=str)

engine_src = inspect.getsource(run_pf)
module_text = "# Standalone PF engine exported by 10_finetune. Vendored by submission.ipynb.\n"
module_text += "import json\nimport numpy as np\nimport pandas as pd\n\n"
module_text += engine_src + "\n\n"
module_text += "MODEL = json.loads('''" + json.dumps(MODEL_CARD) + "''')\n\n"
module_text += """
def predict_well(hz, tw):
    # hz: horizontal df with MD,X,Y,Z,GR and TVT_input (NaN on eval rows)
    #     (a train-style df with full TVT also works: pass TVT as TVT_input)
    # tw: typewell df with TVT, GR
    hz = hz.sort_values("MD").reset_index(drop=True)
    tw_s = tw.sort_values("TVT")
    twt = tw_s["TVT"].values.astype(float)
    twg = tw_s["GR"].ffill().bfill().values.astype(float)
    Z = hz["Z"].values.astype(float); MD = hz["MD"].values.astype(float)
    gr = pd.Series(hz["GR"].values).interpolate(limit_direction="both")\\
           .fillna(90.0).values
    ti = hz["TVT_input"].values.astype(float)
    known = np.isfinite(ti)
    kn = np.where(known)[0]; ev = np.where(~known)[0]
    if len(ev) == 0:
        return ti.copy()
    tvt = ti.copy()
    tvt[~known] = ti[kn[-1]]          # placeholder; engine reads tvt only on kn
    comp = []
    for name, params in MODEL["configs"].items():
        runs = [run_pf(twt, twg, tvt, Z, MD, gr, kn, ev, seed=s, **params)
                for s in MODEL["seeds"]]
        comp.append(np.mean(runs, axis=0))
    X = np.column_stack(comp)
    if MODEL["combiner"] == "uniform":
        p = X.mean(1)
    elif MODEL["combiner"] == "median":
        p = np.median(X, 1)
    else:
        q = MODEL["trimmed_q"]
        lo, hi = np.quantile(X, [q, 1 - q], axis=1)
        p = np.clip(X, lo[:, None], hi[:, None]).mean(1)
    pp = MODEL.get("postproc", {})
    if pp.get("hold_w", 0) > 0:
        p = (1 - pp["hold_w"]) * p + pp["hold_w"] * ti[kn[-1]]
    if pp.get("anchor_tau", 0) > 0:
        t = np.arange(len(p), dtype=float)
        p = p + (ti[kn[-1]] - p[0]) * np.exp(-t / pp["anchor_tau"])
    if pp.get("med_win", 0) > 1:
        p = pd.Series(p).rolling(int(pp["med_win"]), center=True,
                                 min_periods=1).median().values
    out = ti.copy(); out[ev] = p
    return out
"""
(EXPORT_DIR / "pf_engine.py").write_text(module_text)
print("wrote export/model_card.json and export/pf_engine.py")

# ---- integrity test on a simulated test-style well ----
import importlib.util

spec = importlib.util.spec_from_file_location("pf_engine", EXPORT_DIR / "pf_engine.py")
eng = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eng)

wid = COMMON[0]
hz, tw = load_pair(wid)
fr = mask_frac_for(wid)
m = tail_mask(len(hz), fr)
hz_test = hz.copy()
hz_test["TVT_input"] = hz_test["TVT"].where(~m, np.nan)  # simulate test file
full = eng.predict_well(hz_test, tw)
ref = PRED[wid]
assert np.allclose(full[m], ref, atol=1e-4), "export mismatch vs in-notebook prediction"
assert np.allclose(full[~m], hz["TVT"].values[~m], atol=1e-6), "known rows must pass through"
print(
    f"integrity test PASSED on well {wid}: exported engine == notebook "
    f"(eval rmse {rmse(full[m], hz['TVT'].values[m]):.3f})"
)

## Workflow

Tune levers in the control panel → run → evaluate → A/B against the incumbent →
when a variant wins, make it `DESIGN`, re-run, **export**. `submission.ipynb`
always vendors the latest `export/pf_engine.py` + `model_card.json`; the
integrity test guarantees the exported artifact is exactly what was benchmarked.